In [1]:
import sccellfie
import scanpy as sc
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import glasbey

import textwrap

## To avoid warnings
import warnings
warnings.filterwarnings("ignore")

/home/kvalem/.conda/envs/sccellfie/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/kvalem/.conda/envs/sccellfie/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [2]:
import anndata as ad

In [3]:
adata = ad.read_h5ad("/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/40_gex_surface_prot/single_cell_normal/002_annotate_adata_normal.h5ad")

In [4]:
adata = adata[~adata.obs_names.duplicated(), :]

In [5]:
adata

View of AnnData object with n_obs × n_vars = 18263 × 18038
    obs: 'sample_id', 'condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'rna_leiden', 'doublet_score', 'predicted_doublet', 'rna_leiden_12', 'rna_leiden_2', 'cell_annotation_05'
    var: 'gene_ids', 'feature_types', 'mito', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'n_cells'
    uns: 'cell_annotation_05_colors', 'condition_colors', 'hvg', 'log1p', 'neighbors', 'pca', 'predicted_doublet_colors', 'rna_leiden', 'rna_leiden_12', 'rna_leiden_

In [6]:
import os
import glob
import scvelo as scv
import scanpy as sc

def read_scvelo_2019_looms():
    loom_dir = "/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/trajectory_inference/loom"
    
    # find all loom files starting with 2021
    loom_files = sorted(
        glob.glob(os.path.join(loom_dir, "2019*.loom"))
    )
    
    print(f"Found {len(loom_files)} loom files.")
    
    adatas = []
    
    for file in loom_files:
        print(f"Reading {os.path.basename(file)}")
        
        adata = ad.read_loom(file)
        adata.var_names_make_unique()
        
        # store filename (without extension) as metadata
        sample_name = os.path.basename(file).replace(".loom", "")
        adata.obs["sample_id"] = sample_name
        
        adatas.append(adata)
    
    # concatenate all
    adata_combined = sc.concat(adatas, join="outer", label="batch", keys=[a.obs["sample_id"][0] for a in adatas])
    
    return adata_combined


In [7]:
adata_velocity = read_scvelo_2019_looms()

Found 5 loom files.
Reading 2019_10mix1.loom
Reading 2019_10mix2.loom
Reading 2019_11mix1.loom
Reading 2019_11mix2.loom
Reading 2019_GF1.loom


In [8]:
adata_scvelo = adata_velocity

In [9]:
adata_scvelo

AnnData object with n_obs × n_vars = 24112 × 32285
    obs: 'sample_id', 'batch'
    layers: 'matrix', 'ambiguous', 'spliced', 'unspliced'

In [10]:
adata.obs_names

Index(['AAACCTGAGTCTTGCA-1_10mix1', 'AAACCTGAGTTAAGTG-1_10mix1',
       'AAACCTGCAGGATCGA-1_10mix1', 'AAACCTGCATAACCTG-1_10mix1',
       'AAACCTGCATACCATG-1_10mix1', 'AAACCTGCATTGAGCT-1_10mix1',
       'AAACCTGGTGTGCCTG-1_10mix1', 'AAACCTGTCGCCCTTA-1_10mix1',
       'AAACGGGCAAGGGTCA-1_10mix1', 'AAACGGGCAAGGTTCT-1_10mix1',
       ...
       'TTTATGCCAGCTGCAC-1_GF2', 'TTTCCTCAGGAATGGA-1_GF2',
       'TTTCCTCTCATAGCAC-1_GF2', 'TTTGCGCAGGGAGTAA-1_GF2',
       'TTTGCGCTCAACCATG-1_GF2', 'TTTGCGCTCGGTGTTA-1_GF2',
       'TTTGTCAAGGCTAGCA-1_GF2', 'TTTGTCAAGTGCCAGA-1_GF2',
       'TTTGTCAAGTTGTCGT-1_GF2', 'TTTGTCATCGATGAGG-1_GF2'],
      dtype='object', length=18263)

In [11]:
import re

# Step 1: extract clean barcode
barcodes = (
    adata_scvelo.obs_names
    .str.split(":").str[1]      # take part after :
    .str.replace("x", "", regex=False)  # remove trailing x
)

# Step 2: clean sample names
samples = (
    adata_scvelo.obs["sample_id"]
    .str.replace("2019_", "", regex=False)
)

# Step 3: build matching obs_names
new_names = samples + "_" + barcodes + "-1"

adata_scvelo.obs_names = new_names


In [12]:
adata_scvelo.obs

,sample_id,batch
10mix1_AAAGCAAGTGAGGCTA-1,2019_10mix1,2019_10mix1
10mix1_AACTCAGTCGCGTTTC-1,2019_10mix1,2019_10mix1
10mix1_AAAGTAGGTTCCGTCT-1,2019_10mix1,2019_10mix1
10mix1_AAATGCCAGTAGCCGA-1,2019_10mix1,2019_10mix1
10mix1_AACTCAGCATTTCACT-1,2019_10mix1,2019_10mix1
...,...,...
GF1_TTTGTCACAATCCGAT-1,2019_GF1,2019_GF1
GF1_TTTCCTCGTGTCGCTG-1,2019_GF1,2019_GF1
GF1_TTTCCTCTCTAACCGA-1,2019_GF1,2019_GF1
GF1_TTTGCGCAGTGTGGCA-1,2019_GF1,2019_GF1


In [13]:
adata.obs_names

Index(['AAACCTGAGTCTTGCA-1_10mix1', 'AAACCTGAGTTAAGTG-1_10mix1',
       'AAACCTGCAGGATCGA-1_10mix1', 'AAACCTGCATAACCTG-1_10mix1',
       'AAACCTGCATACCATG-1_10mix1', 'AAACCTGCATTGAGCT-1_10mix1',
       'AAACCTGGTGTGCCTG-1_10mix1', 'AAACCTGTCGCCCTTA-1_10mix1',
       'AAACGGGCAAGGGTCA-1_10mix1', 'AAACGGGCAAGGTTCT-1_10mix1',
       ...
       'TTTATGCCAGCTGCAC-1_GF2', 'TTTCCTCAGGAATGGA-1_GF2',
       'TTTCCTCTCATAGCAC-1_GF2', 'TTTGCGCAGGGAGTAA-1_GF2',
       'TTTGCGCTCAACCATG-1_GF2', 'TTTGCGCTCGGTGTTA-1_GF2',
       'TTTGTCAAGGCTAGCA-1_GF2', 'TTTGTCAAGTGCCAGA-1_GF2',
       'TTTGTCAAGTTGTCGT-1_GF2', 'TTTGTCATCGATGAGG-1_GF2'],
      dtype='object', length=18263)

In [14]:
len(set(adata.obs_names)), len(set(adata_scvelo.obs_names)), len(
    set(adata.obs_names) & set(adata_scvelo.obs_names)
)

(18263, 24112, 0)

In [15]:
adata_scvelo = scv.utils.merge(adata_scvelo, adata)

In [16]:
scv.pp.filter_and_normalize(adata_scvelo)

Normalized count data: X, spliced, unspliced.


In [17]:
# 1. Get the shared cell barcodes in the exact order of your scVelo object
scvelo_cells = adata_scvelo.obs_names

# 2. Extract the integer positional indices of these cells from the main adata object
cell_indices = adata.obs_names.get_indexer(scvelo_cells)

# 3. Slice the connectivities and distances matrices across both dimensions (rows and columns)
adata_scvelo.obsp['connectivities'] = adata.obsp['connectivities'][cell_indices, :][:, cell_indices]
adata_scvelo.obsp['distances'] = adata.obsp['distances'][cell_indices, :][:, cell_indices]

# 4. (Optional) Verify the slot update
print(adata_scvelo)

AnnData object with n_obs × n_vars = 11351 × 18038
    obs: 'sample_id', 'batch', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'sample_batch', 'condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'rna_leiden', 'doublet_score', 'predicted_doublet', 'rna_leiden_12', 'rna_leiden_2', 'cell_annotation_05', 'n_counts'
    var: 'gene_ids', 'feature_types', 'mito', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'n_cells', 'gene_count_corr'
    uns: 'cell_annotation_05_colors', 'condition_co

In [18]:
adata_scvelo

AnnData object with n_obs × n_vars = 11351 × 18038
    obs: 'sample_id', 'batch', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'sample_batch', 'condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'rna_leiden', 'doublet_score', 'predicted_doublet', 'rna_leiden_12', 'rna_leiden_2', 'cell_annotation_05', 'n_counts'
    var: 'gene_ids', 'feature_types', 'mito', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'n_cells', 'gene_count_corr'
    uns: 'cell_annotation_05_colors', 'condition_co

In [19]:
adata_scvelo_effector = adata_scvelo[adata_scvelo.obs["condition"].isin(["effector"])]

In [20]:
adata_scvelo = adata_scvelo_effector

In [21]:
adata_scvelo

View of AnnData object with n_obs × n_vars = 4754 × 18038
    obs: 'sample_id', 'batch', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'sample_batch', 'condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'rna_leiden', 'doublet_score', 'predicted_doublet', 'rna_leiden_12', 'rna_leiden_2', 'cell_annotation_05', 'n_counts'
    var: 'gene_ids', 'feature_types', 'mito', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'n_cells', 'gene_count_corr'
    uns: 'cell_annotation_05_colors', 'condi

In [22]:
adata_scvelo.X = adata_scvelo.layers['spliced'].copy()

In [23]:
results = sccellfie.run_sccellfie_pipeline(adata_scvelo,
                                           organism='mouse',
                                           sccellfie_data_folder=None,
                                           n_counts_col=None, # Total counts per cell will be computed if left as None,
                                           process_by_group=False, # Whether to do the processing by cell groups
                                           groupby=None, # Column indicating cell groups if `process_by_group=True`
                                           neighbors_key='neighbors', # Neighbors information if precomputed. Otherwise, it will be computed here
                                           n_neighbors=10, # Number of neighbors to use
                                           batch_key=None, # there is no batch_key in this dataset
                                           threshold_key='sccellfie_threshold',  # This is for using the default database. If personalized thresholds are used, specificy column name
                                           smooth_cells=True, # Whether to perform gene expression smoothing before running the tool
                                           alpha=0.33, # Importance of neighbors' expression for the smoothing (0 to 1)
                                           chunk_size=5000, # Number of chunks to run the processing steps (helps with the memory)
                                           disable_pbar=False,
                                           save_folder=None, # In case results will be saved. If so, results will not be returned and should be loaded from the folder (see sccellfie.io.load_data function
                                           save_filename=None # Name for saving the files, otherwise a default name will be used
                                          )


==== scCellFie Pipeline: Initializing ====
Loading scCellFie database for organism: mouse

==== scCellFie Pipeline: Processing entire dataset ====

---- scCellFie Step: Preprocessing data ----

---- scCellFie Step: Preparing inputs ----
Gene names corrected to match database: 9
Shape of new adata object: (4754, 682)
Number of GPRs: 623
Shape of tasks by genes: (193, 682)
Shape of reactions by genes: (623, 682)
Shape of tasks by reactions: (193, 623)

---- scCellFie Step: Smoothing gene expression ----


Smoothing Expression: 100%|██████████| 1/1 [00:00<00:00, 52.59it/s]



---- scCellFie Step: Computing gene scores ----

---- scCellFie Step: Computing reaction activity ----


Cell Rxn Activities: 100%|██████████| 4754/4754 [00:31<00:00, 151.06it/s]



---- scCellFie Step: Computing metabolic task activity ----
Removed 13 metabolic tasks with zeros across all cells.

==== scCellFie Pipeline: Processing completed successfully ====


In [24]:
results.keys()

dict_keys(['adata', 'gpr_rules', 'task_by_gene', 'rxn_by_gene', 'task_by_rxn', 'rxn_info', 'task_info', 'thresholds', 'organism'])

In [25]:
sccellfie.io.save_adata(adata=results['adata'], output_directory='/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/prj_honda_15062026/scCellFie', filename='Honda_scCellFie_healthy_effector')

/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/prj_honda_15062026/scCellFie/Honda_scCellFie_healthy_effector.h5ad was correctly saved
/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/prj_honda_15062026/scCellFie/Honda_scCellFie_healthy_effector_reactions.h5ad was correctly saved
/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/prj_honda_15062026/scCellFie/Honda_scCellFie_healthy_effector_metabolic_tasks.h5ad was correctly saved
